In [ ]:
!pip install ollama

In [ ]:
import json
from ollama import Client

In [ ]:
OLLAMA_API_KEY = "OLLAMA_API_KEY"

In [ ]:
with open('/content/CyberMetric-500-v1.json', 'r') as f:
    file = json.load(f)

In [ ]:
client = Client(
    host='https://https://ollama.com',
    headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}
)

In [ ]:
translation_prompt = """Aşağıdaki siber güvenlik sorusunu ve şıklarını Türkçeye çevir.
KURALLAR:
- Teknik terimleri (protokol isimleri, saldırı türleri, kısaltmalar: TCP/IP, DDoS, Man-in-the-Middle, buffer overflow vb.) İNGİLİZCE bırak
- Sadece cümle yapısını ve genel ifadeleri Türkçeye çevir
- Doğru cevabın harfini (A/B/C/D) DEĞİŞTİRME
- Çıktıyı SADECE JSON olarak ver, başka hiçbir şey yazma

Girdi:
{soru_json}

Çıktı formatı:
{{"question": "...", "answers": {{"A": "...", "B": "...", "C": "...", "D": "..."}}, "solution": "X"}}"""

In [ ]:
import requests
import time
def translate_item(item, model="gpt-oss:120b"):
    prompt = translation_prompt.format(soru_json=json.dumps(item, ensure_ascii=False))
    resp = requests.post("https://ollama.com/api/generate",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
    json={"model": "gpt-oss:120b", "prompt": f"{prompt}", "stream": False})
    data = resp.json()
    return data

translated = []
for i, item in enumerate(file['questions']):  # Corrected: iterate over file['questions']
    try:
        t = translate_item(item)
        translated.append(t)
        print(translated[i])
    except Exception as e:
        print(f"hata: {e}, orijinal atlandı")
    time.sleep(0.5)  # rate limit için

with open("cybermetric_tr.jsonl", "w", encoding="utf-8") as f:
    for t in translated:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

{'model': 'gpt-oss:120b', 'created_at': '2026-07-25T22:28:28.549054403Z', 'response': '{"question":"Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?","answers":{"A":"Kalıcı","B":"Transfer edilebilirlik","C":"Tekdüzelik","D":"Bağışlama"},"solution":"A"}', 'thinking': 'We need to translate the question and answers, keeping technical terms in English. The question: "Which of the following is a desirable property of a biometric system?" There\'s no technical term, can translate fully.\n\nAnswers: A: "Permanent", B: "Transferability", C: "Uniformity", D: "Forgiveness". These are English words; they are not technical terms? They are generic. The rule says keep technical terms like protocol names etc. But these are not technical terms; they are ordinary English words, but the instruction says only to translate sentences and general expressions, keep technical terms. So we can translate these into Turkish. But they are single words, not technical. Should we translate them? 

In [ ]:
translated = []
with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            t = json.loads(line)
            translated.append(t)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

In [ ]:
# Overwrite the cybermetric_tr.jsonl file with the new, flat JSONL format
with open("cybermetric_tr.jsonl", "w", encoding="utf-8") as f:
    for t in formatted_translated_data:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print("cybermetric_tr.jsonl successfully updated with the desired format.")

cybermetric_tr.jsonl successfully updated with the desired format.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("sk75/qwen3.5-4b-siber-guvenlik-tr-merged")
model = AutoModelForCausalLM.from_pretrained("sk75/qwen3.5-4b-siber-guvenlik-tr-merged", device_map="auto")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-4B", device_map="auto")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [ ]:
def build_prompt(item):
    return f"""Aşağıdaki çoktan seçmeli siber güvenlik sorusunu cevapla. SADECE doğru şıkkın harfini yaz (A, B, C veya D). Açıklama yapma, başka hiçbir şey yazma.

Soru: {item['question']}
A) {item['answers']['A']}
B) {item['answers']['B']}
C) {item['answers']['C']}
D) {item['answers']['D']}

Cevap:"""

In [ ]:
import re

def ask_model(item, model, tokenizer):
    prompt = build_prompt(item)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    out = model.generate(
        **inputs,
        max_new_tokens=3,        # sadece harf beklediğimiz için kısa tut
        do_sample=False,         # deterministik cevap için (temperature yok)
        pad_token_id=tokenizer.eos_token_id
    )

    generated = out[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    match = re.search(r'[ABCD]', text.upper())
    return match.group(0) if match else None

In [ ]:
import requests
def ask_ollama_model(item, model="gpt-oss:120b"):
    prompt = build_prompt(item)
    resp = requests.post("https://ollama.com/api/generate",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
    json={"model": model, "prompt": f"{prompt}", "stream": False})
    data = resp.json()
    return data["response"]

In [ ]:
import json
with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0
results = []

for item in test_data:
    pred = ask_ollama_model(item)
    is_correct = (pred == item['solution'])
    correct += is_correct
    total += 1
    print(f"Soru: {item['question']}")
    print(f"Tahmin: {pred}")
    print(f"Doğru Cevap: {item['solution']}")
    print(f"Doğru Mu: {is_correct}\n")
    results.append({"question": item['question'], "predicted": pred, "actual": item['solution'], "correct": is_correct})

print(f"Doğruluk: {correct}/{total} = {correct/total*100:.2f}%")

Soru: Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: TCP/IP ağında, bir paket içinde ağ adreslerini ve yönlendirme bilgilerini tutmak için kullanılan protokol hangisidir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Kişisel gizlilik politikalarında beklenmedik olumsuz sonuçlar bağlamında, özel bilgilerin saklama süresi konusunda sağlayıcı ve tüketici üzerinde hangi sorumluluk vardır?
Tahmin: D
Doğru Cevap: A
Doğru Mu: False

Soru: Hangi saldırı türü, saldırganın iki iletişim sistemi arasında bir store-and-forward veya proxy mekanizması gibi davranmasını içerir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Bir kuruluşun güvenlik önlemlerinde logging ve monitoring'in temel amacı nedir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Afet kurtarma testlerinde yapılandırılmış yürütmelerin faydası nedir?
Tahmin: C
Doğru Cevap: C
Doğru Mu: True

Soru: Hangi güvenlik süreci ölçütü, bir veritabanı sunucusu için uygun bir y

In [ ]:
import json

with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0
results = []

for item in test_data:
    pred = ask_ollama_model(item, model="gemma4:cloud")
    is_correct = (pred == item['solution'])
    correct += is_correct
    total += 1
    print(f"Soru: {item['question']}")
    print(f"Tahmin: {pred}")
    print(f"Doğru Cevap: {item['solution']}")
    print(f"Doğru Mu: {is_correct}\n")
    results.append({"question": item['question'], "predicted": pred, "actual": item['solution'], "correct": is_correct})

print(f"Doğruluk: {correct}/{total} = {correct/total*100:.2f}%")

Soru: Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: TCP/IP ağında, bir paket içinde ağ adreslerini ve yönlendirme bilgilerini tutmak için kullanılan protokol hangisidir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Kişisel gizlilik politikalarında beklenmedik olumsuz sonuçlar bağlamında, özel bilgilerin saklama süresi konusunda sağlayıcı ve tüketici üzerinde hangi sorumluluk vardır?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: Hangi saldırı türü, saldırganın iki iletişim sistemi arasında bir store-and-forward veya proxy mekanizması gibi davranmasını içerir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Bir kuruluşun güvenlik önlemlerinde logging ve monitoring'in temel amacı nedir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Afet kurtarma testlerinde yapılandırılmış yürütmelerin faydası nedir?
Tahmin: A
Doğru Cevap: C
Doğru Mu: False

Soru: Hangi güvenlik süreci ölçütü, bir veritabanı sunucusu için uygun bir y

In [ ]:
with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0
results = []

for item in test_data:
    pred = ask_ollama_model(item, model="gemma4:31b-cloud")
    is_correct = (pred == item['solution'])
    correct += is_correct
    total += 1
    print(f"Soru: {item['question']}")
    print(f"Tahmin: {pred}")
    print(f"Doğru Cevap: {item['solution']}")
    print(f"Doğru Mu: {is_correct}\n")
    results.append({"question": item['question'], "predicted": pred, "actual": item['solution'], "correct": is_correct})

print(f"Doğruluk: {correct}/{total} = {correct/total*100:.2f}%")

Soru: Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: TCP/IP ağında, bir paket içinde ağ adreslerini ve yönlendirme bilgilerini tutmak için kullanılan protokol hangisidir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Kişisel gizlilik politikalarında beklenmedik olumsuz sonuçlar bağlamında, özel bilgilerin saklama süresi konusunda sağlayıcı ve tüketici üzerinde hangi sorumluluk vardır?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: Hangi saldırı türü, saldırganın iki iletişim sistemi arasında bir store-and-forward veya proxy mekanizması gibi davranmasını içerir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Bir kuruluşun güvenlik önlemlerinde logging ve monitoring'in temel amacı nedir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Afet kurtarma testlerinde yapılandırılmış yürütmelerin faydası nedir?
Tahmin: A
Doğru Cevap: C
Doğru Mu: False

Soru: Hangi güvenlik süreci ölçütü, bir veritabanı sunucusu için uygun bir y

In [ ]:
import json

with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0
results = []

for item in test_data:
    pred = ask_ollama_model(item, model="nemotron-3-nano:30b-cloud")
    is_correct = (pred == item['solution'])
    correct += is_correct
    total += 1
    print(f"Soru: {item['question']}")
    print(f"Tahmin: {pred}")
    print(f"Doğru Cevap: {item['solution']}")
    print(f"Doğru Mu: {is_correct}\n")
    results.append({"question": item['question'], "predicted": pred, "actual": item['solution'], "correct": is_correct})

print(f"Doğruluk: {correct}/{total} = {correct/total*100:.2f}%")

Soru: Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?
Tahmin: C
Doğru Cevap: A
Doğru Mu: False

Soru: TCP/IP ağında, bir paket içinde ağ adreslerini ve yönlendirme bilgilerini tutmak için kullanılan protokol hangisidir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Kişisel gizlilik politikalarında beklenmedik olumsuz sonuçlar bağlamında, özel bilgilerin saklama süresi konusunda sağlayıcı ve tüketici üzerinde hangi sorumluluk vardır?
Tahmin: D
Doğru Cevap: A
Doğru Mu: False

Soru: Hangi saldırı türü, saldırganın iki iletişim sistemi arasında bir store-and-forward veya proxy mekanizması gibi davranmasını içerir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Bir kuruluşun güvenlik önlemlerinde logging ve monitoring'in temel amacı nedir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Afet kurtarma testlerinde yapılandırılmış yürütmelerin faydası nedir?
Tahmin: C
Doğru Cevap: C
Doğru Mu: True

Soru: Hangi güvenlik süreci ölçütü, bir veritabanı sunucusu için uygun bir 

In [ ]:
import json
with open("cybermetric_tr.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

correct = 0
total = 0
results = []

for item in test_data:
    pred = ask_model(item, model, tokenizer)
    is_correct = (pred == item['solution'])
    correct += is_correct
    total += 1
    print(f"Soru: {item['question']}")
    print(f"Tahmin: {pred}")
    print(f"Doğru Cevap: {item['solution']}")
    print(f"Doğru Mu: {is_correct}\n")
    results.append({"question": item['question'], "predicted": pred, "actual": item['solution'], "correct": is_correct})

print(f"Doğruluk: {correct}/{total} = {correct/total*100:.2f}%")

Soru: Aşağıdakilerden hangisi bir biyometrik sistemin istenen özelliğidir?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: TCP/IP ağında, bir paket içinde ağ adreslerini ve yönlendirme bilgilerini tutmak için kullanılan protokol hangisidir?
Tahmin: C
Doğru Cevap: B
Doğru Mu: False

Soru: Kişisel gizlilik politikalarında beklenmedik olumsuz sonuçlar bağlamında, özel bilgilerin saklama süresi konusunda sağlayıcı ve tüketici üzerinde hangi sorumluluk vardır?
Tahmin: A
Doğru Cevap: A
Doğru Mu: True

Soru: Hangi saldırı türü, saldırganın iki iletişim sistemi arasında bir store-and-forward veya proxy mekanizması gibi davranmasını içerir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Bir kuruluşun güvenlik önlemlerinde logging ve monitoring'in temel amacı nedir?
Tahmin: B
Doğru Cevap: B
Doğru Mu: True

Soru: Afet kurtarma testlerinde yapılandırılmış yürütmelerin faydası nedir?
Tahmin: A
Doğru Cevap: C
Doğru Mu: False

Soru: Hangi güvenlik süreci ölçütü, bir veritabanı sunucusu için uygun bir 

Bütün modellerin testi 500 adet siber güvenlik sorusu ile test edilmiştir. Dataset: https://huggingface.co/datasets/sk75/CyberEval500_TR

Görüldüğü üzere model benchmark sonuçları şunlardır:

* GPT-OSS-120B -> %93
* Gemma4:31B -> %94
* nemotron-3-nano:30B -> %90.40
* sk75/qwen3.5-4b-siber-guvenlik-tr-merged finetuned -> %81